In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    mean_absolute_percentage_error
)
from sklearn.inspection import permutation_importance
import xgboost as xgb
from xgboost import XGBRegressor, plot_importance
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# SHAP (pre-installed on Kaggle)
try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("SHAP not available – skipping SHAP plots")

# ── Plot style ────────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor": "#0f1117",
    "axes.facecolor":   "#1a1d27",
    "axes.edgecolor":   "#2e3250",
    "axes.labelcolor":  "#c8cfe8",
    "xtick.color":      "#8892b0",
    "ytick.color":      "#8892b0",
    "text.color":       "#c8cfe8",
    "grid.color":       "#1e2340",
    "grid.linewidth":   0.6,
    "font.family":      "DejaVu Sans",
    "figure.dpi":       120,
})
ACCENT   = "#64ffda"
ACCENT2  = "#f472b6"
ACCENT3  = "#fbbf24"
BLUE     = "#38bdf8"
PURPLE   = "#818cf8"

# ── 1. Load data ──────────────────────────────────────────────
df = pd.read_csv("/kaggle/input/datasets/bapdesilva/betel-ds/betel_yield_dataset_40000_v2.csv")

print("=" * 55)
print("  DATASET OVERVIEW")
print("=" * 55)
print(f"  Shape         : {df.shape}")
print(f"  Missing values: {df.isnull().sum().sum()}")
print(f"  Target range  : {df['Leaf_Count_per_Bed'].min():.0f} – {df['Leaf_Count_per_Bed'].max():.0f}")
print(f"  Target mean   : {df['Leaf_Count_per_Bed'].mean():.2f} ± {df['Leaf_Count_per_Bed'].std():.2f}")
print("=" * 55)
print(df.describe().round(2))


# ── 2. Feature engineering ────────────────────────────────────
def engineer_features(df):
    df = df.copy()

    # Interaction terms (domain-informed)
    df["Moisture_x_Rain"]    = df["Soil_Moisture"] * df["Rainfall"]
    df["Moisture_x_Humid"]   = df["Soil_Moisture"] * df["Humidity"]
    df["NPK_sum"]            = df["N"] + df["P"] + df["K"]
    df["NPK_product"]        = df["N"] * df["P"] * df["K"]
    df["Temp_x_Humid"]       = df["Temperature"] * df["Humidity"]
    df["Rain_x_Humid"]       = df["Rainfall"] * df["Humidity"]
    df["pH_x_NPK"]           = df["Soil_pH"] * df["NPK_sum"]
    df["Moisture_sq"]        = df["Soil_Moisture"] ** 2
    df["Rain_sq"]            = df["Rainfall"] ** 2

    # Log transforms (reduces skew on rain)
    df["log_Rain"]           = np.log1p(df["Rainfall"])
    df["log_NPK"]            = np.log1p(df["NPK_sum"])

    # Ratios
    df["N_to_P"]             = df["N"] / (df["P"] + 1e-6)
    df["Moisture_to_Rain"]   = df["Soil_Moisture"] / (df["Rainfall"] + 1e-6)

    # Optimal range flags (domain heuristics for betel)
    df["optimal_moisture"]   = ((df["Soil_Moisture"] >= 55) & (df["Soil_Moisture"] <= 75)).astype(int)
    df["optimal_temp"]       = ((df["Temperature"]   >= 25) & (df["Temperature"]   <= 32)).astype(int)
    df["optimal_pH"]         = ((df["Soil_pH"]       >= 5.5) & (df["Soil_pH"]      <= 7.0)).astype(int)
    df["optimal_rain"]       = ((df["Rainfall"]      >= 150) & (df["Rainfall"]     <= 300)).astype(int)

    return df

df_eng = engineer_features(df)

FEATURES = [c for c in df_eng.columns if c != "Leaf_Count_per_Bed"]
TARGET   = "Leaf_Count_per_Bed"

X = df_eng[FEATURES]
y = df_eng[TARGET]

print(f"\n  Total features after engineering: {len(FEATURES)}")

# ── 3. Train / validation / test split ───────────────────────
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.10, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.111, random_state=42  # ~10% of total
)
print(f"\n  Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

# ── 4. Optuna hyperparameter search ──────────────────────────
print("\n  Running Optuna hyperparameter search (120 trials)…")

def objective(trial):
    params = {
        "n_estimators":       trial.suggest_int("n_estimators", 400, 1200),
        "max_depth":          trial.suggest_int("max_depth", 3, 9),
        "learning_rate":      trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "subsample":          trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":   trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "colsample_bylevel":  trial.suggest_float("colsample_bylevel", 0.5, 1.0),
        "min_child_weight":   trial.suggest_int("min_child_weight", 1, 10),
        "reg_alpha":          trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda":         trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        "gamma":              trial.suggest_float("gamma", 0.0, 5.0),
        "random_state":       42,
        "tree_method":        "hist",
        "device":             "cuda",    # uses GPU on Kaggle – falls back to cpu
        "eval_metric":        "rmse",
        "early_stopping_rounds": 40,
        "verbosity":          0,
    }
    model = XGBRegressor(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )
    preds = model.predict(X_val)
    return r2_score(y_val, preds)

study = optuna.create_study(direction="maximize",
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=120, show_progress_bar=True)

best_params = study.best_params
best_params.update({
    "random_state": 42,
    "tree_method":  "hist",
    "device":       "cuda",
    "eval_metric":  ["rmse", "mae"],
    "early_stopping_rounds": 50,
    "verbosity":    0,
})
print(f"\n  Best trial R²  : {study.best_value:.6f}")
print(f"  Best params    : {best_params}")

# ── 5. Train final model with learning curves ─────────────────
print("\n  Training final model…")

model = XGBRegressor(**best_params)
model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=False,
)

evals = model.evals_result()
train_rmse = evals["validation_0"]["rmse"]
val_rmse   = evals["validation_1"]["rmse"]
train_mae  = evals["validation_0"]["mae"]
val_mae    = evals["validation_1"]["mae"]

# Predictions
y_train_pred = model.predict(X_train)
y_val_pred   = model.predict(X_val)
y_test_pred  = model.predict(X_test)

def regression_report(y_true, y_pred, split_name):
    r2   = r2_score(y_true, y_pred)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    print(f"\n  ── {split_name} ──")
    print(f"    R²   : {r2:.6f}  ({r2*100:.2f}%)")
    print(f"    MAE  : {mae:.4f} leaves")
    print(f"    RMSE : {rmse:.4f} leaves")
    print(f"    MAPE : {mape:.2f}%")
    return {"R2": r2, "MAE": mae, "RMSE": rmse, "MAPE": mape}

print("\n" + "=" * 55)
print("  FINAL MODEL PERFORMANCE")
print("=" * 55)
tr_metrics  = regression_report(y_train, y_train_pred, "Train")
val_metrics = regression_report(y_val,   y_val_pred,   "Validation")
te_metrics  = regression_report(y_test,  y_test_pred,  "Test")
